In [1]:
%%writefile main.py
"""
Orbit Wars - "The Banking & Beachhead" Agent
Features:
- Exact Frame-by-Frame Physics Raycaster (Zero sun deaths, zero deep-space misses).
- Logistics Banking System (Pipes ships from safe Rear planets to Frontline Hubs).
- Cluster Beachhead Targeting (Cracks enemy clusters by anchoring on mid-level targets).
- Consolidation (Pump & Dump evacuation of doomed planets to prevent fleet splitting).
"""

import math
import time
from collections import defaultdict, namedtuple

# ============================================================
# Core Configuration
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5
HORIZON = 150
TOTAL_STEPS = 500

# Banking & Logistics
BANKING_SAFE_DISTANCE = 38.0  # Distance from enemies to be considered a "Rear Spoke"
BANKING_MIN_GARRISON = 8      # Ships kept on Rear planets
CLUSTER_RADIUS = 22.0         # Distance to group planets into clusters

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet = namedtuple("Fleet",["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

# ============================================================
# Exact Frame-Perfect Physics Engine
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    ratio = max(0.0, min(1.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

class PlanetTracker:
    """Precomputes exact positions of all planets for the next 150 frames to ensure zero misses."""
    def __init__(self, planets, initial_planets, ang_vel, comets, comet_ids):
        self.pos = {}
        for p in planets:
            self.pos[p.id] =[self._calc(p, t, ang_vel, initial_planets, comets, comet_ids) for t in range(HORIZON + 1)]
            
    def _calc(self, p, t, ang_vel, initial_planets, comets, comet_ids):
        if p.id in comet_ids:
            for c in comets:
                if p.id in c.get("planet_ids",[]):
                    idx = c["planet_ids"].index(p.id)
                    f_idx = c.get("path_index", 0) + t
                    if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]):
                        return c["paths"][idx][f_idx]
            return None
        init = initial_planets.get(p.id)
        if not init: return (p.x, p.y)
        r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
        if r + init.radius >= 50.0: return (p.x, p.y) # Static
        ang = math.atan2(p.y - CENTER_Y, p.x - CENTER_X) + ang_vel * t
        return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight_raycast(src, tgt, ships, tracker):
    """
    Frame-perfect raycaster. Validates that the fleet intersects the exact bounding box 
    of the moving planet without clipping the sun.
    """
    speed = fleet_speed(ships)
    sx, sy, sr = src.x, src.y, src.radius
    tr = tgt.radius
    
    for T in range(1, HORIZON):
        pos = tracker.pos[tgt.id][T]
        if not pos: continue
        tx, ty = pos
        
        dist_centers = math.hypot(tx - sx, ty - sy)
        # Fleet stops at the outer edge of the planet's radius
        flight_dist = max(0.0, dist_centers - sr - 0.1 - tr)
        
        # Does the fleet reach the planet exactly on Turn T?
        if (T - 1) * speed <= flight_dist <= T * speed:
            angle = math.atan2(ty - sy, tx - sx)
            
            # Verify Sun Avoidance along the exact segment
            start_x = sx + math.cos(angle) * (sr + 0.1)
            start_y = sy + math.sin(angle) * (sr + 0.1)
            end_x = start_x + math.cos(angle) * flight_dist
            end_y = start_y + math.sin(angle) * flight_dist
            
            if point_to_segment_dist(CENTER_X, CENTER_Y, start_x, start_y, end_x, end_y) <= SUN_R + SUN_SAFETY:
                continue # Path hits the sun, abort.
                
            return angle, T
    return None, None

def get_active_fleet_arrival(fleet, tracker, planets):
    """Tracks existing fleets using the exact engine rules to maintain perfect internal ledgers."""
    speed = fleet_speed(fleet.ships)
    dx, dy = math.cos(fleet.angle), math.sin(fleet.angle)
    fx, fy = fleet.x, fleet.y
    
    for N in range(1, HORIZON):
        seg_sx = fx + dx * speed * (N - 1)
        seg_sy = fy + dy * speed * (N - 1)
        seg_ex = fx + dx * speed * N
        seg_ey = fy + dy * speed * N
        
        # Sun / Bounds Check
        if point_to_segment_dist(CENTER_X, CENTER_Y, seg_sx, seg_sy, seg_ex, seg_ey) <= SUN_R: return None, None
        if not (0 <= seg_ex <= 100 and 0 <= seg_ey <= 100): return None, None
            
        best_d = float('inf')
        hit_p = None
        
        # Step 5 Collision (Moving into planet's old space)
        for p in planets:
            pos = tracker.pos[p.id][N - 1]
            if pos and point_to_segment_dist(pos[0], pos[1], seg_sx, seg_sy, seg_ex, seg_ey) <= p.radius:
                d = math.hypot(pos[0] - seg_sx, pos[1] - seg_sy)
                if d < best_d:
                    best_d, hit_p = d, p
        if hit_p: return hit_p.id, N
        
        # Step 6 Collision (Planet sweeps into fleet's endpoint)
        for p in planets:
            pos = tracker.pos[p.id][N]
            if pos and math.hypot(pos[0] - seg_ex, pos[1] - seg_ey) <= p.radius:
                return p.id, N
    return None, None

# ============================================================
# Logistics & Timeline Simulation
# ============================================================
def simulate_timeline(pid, planet, active_fleets, player):
    by_turn = defaultdict(int)
    for f in active_fleets:
        if f['target'] == pid:
            by_turn[f['eta']] += f['ships'] if f['owner'] == player else -f['ships']
            
    curr_owner = planet.owner
    curr_ships = float(planet.ships)
    min_owned_ships = curr_ships if curr_owner == player else 0
    fall_turn = None
    
    for t in range(1, 30):
        if curr_owner != -1: curr_ships += planet.production
        if t in by_turn:
            net_arrival = by_turn[t]
            curr_ships += net_arrival
            if curr_ships < 0:
                curr_owner = -1 if curr_owner == player else player
                curr_ships = -curr_ships
                if fall_turn is None and curr_owner != player:
                    fall_turn = t
                    
        if curr_owner == player:
            min_owned_ships = min(min_owned_ships, curr_ships)
            
    return curr_owner, curr_ships, fall_turn, max(0, int(min_owned_ships))

# ============================================================
# Main Agent Engine
# ============================================================
def agent(obs, config=None):
    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    raw_planets = obs.get("planets",[]) if is_dict else getattr(obs, "planets", [])
    raw_fleets = obs.get("fleets",[]) if is_dict else getattr(obs, "fleets",[])
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    raw_init = obs.get("initial_planets",[]) if is_dict else getattr(obs, "initial_planets", [])
    comets = obs.get("comets",[]) if is_dict else getattr(obs, "comets",[])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids", []))
    
    planets =[Planet(*p) for p in raw_planets]
    fleets = [Fleet(*f) for f in raw_fleets]
    init_map = {Planet(*p).id: Planet(*p) for p in raw_init}
    
    tracker = PlanetTracker(planets, init_map, ang_vel, comets, comet_ids)
    
    # 1. Build Exact Arrival Ledger
    active_flights =[]
    for f in fleets:
        tgt_id, eta = get_active_fleet_arrival(f, tracker, planets)
        if tgt_id is not None:
            active_flights.append({'target': tgt_id, 'eta': eta, 'owner': f.owner, 'ships': f.ships})

    my_planets =[p for p in planets if p.owner == player]
    enemy_planets = [p for p in planets if p.owner not in (-1, player)]
    if not my_planets: return []

    moves =[]
    budget = {}
    
    # 2. State & Role Classification (Hubs vs Spokes)
    hubs = []
    spokes =[]
    
    for p in my_planets:
        owner_at_end, ships_at_end, fall_turn, min_ships = simulate_timeline(p.id, p, active_flights, player)
        
        # Consolidation Rule: Evacuate Doomed Planets
        if fall_turn is not None and min_ships <= 0:
            budget[p.id] = int(p.ships) # Pump and Dump!
            spokes.append(p)
            continue
            
        budget[p.id] = min_ships # Only spend what is guaranteed to survive
        
        dist_to_front = min([math.hypot(p.x - e.x, p.y - e.y) for e in enemy_planets]) if enemy_planets else 999
        if dist_to_front > BANKING_SAFE_DISTANCE:
            spokes.append(p)
        else:
            hubs.append(p)
            
    if not hubs and my_planets: hubs =[max(my_planets, key=lambda x: x.ships)]

    # 3. The Banking System (Route Rear Capital to Frontline Hubs)
    for sp in spokes:
        avail = budget[sp.id] - BANKING_MIN_GARRISON
        if avail > 10:
            # Find closest safe Hub to pipe ships to
            best_hub = min(hubs, key=lambda h: math.hypot(sp.x - h.x, sp.y - h.y))
            if best_hub.id != sp.id:
                angle, t = plan_flight_raycast(sp, best_hub, avail, tracker)
                if angle is not None:
                    moves.append([sp.id, angle, avail])
                    budget[sp.id] -= avail
                    # Add to ledger so the Hub knows reinforcements are coming
                    active_flights.append({'target': best_hub.id, 'eta': t, 'owner': player, 'ships': avail})

    # 4. Strike Coordination (Targeting Beachheads & Enemy Clusters)
    roi_matrix =[]
    for hub in hubs:
        avail = budget[hub.id]
        if avail < 10: continue
        
        for tgt in planets:
            if tgt.owner == player: continue
            
            proj_owner, proj_ships, _, _ = simulate_timeline(tgt.id, tgt, active_flights, player)
            
            # Predict intercept to find ETA
            angle, eta = plan_flight_raycast(hub, tgt, avail, tracker)
            if not angle or eta > 45: continue # Ignore long, deep-space snipes
            
            # Calculate cost factoring in production during flight
            cost = proj_ships + (tgt.production * eta if proj_owner != -1 else 0) + 2
            if cost >= avail: continue
            
            # Extract Cluster & Beachhead Multipliers
            neighbors = sum(1 for op in planets if op.id != tgt.id and math.hypot(op.x - tgt.x, op.y - tgt.y) < CLUSTER_RADIUS)
            cluster_bonus = 1.0 + (neighbors * 0.15)
            
            # Beachhead Domino: Target mid-level planets inside clusters to establish a stronghold
            beachhead_bonus = 1.0
            if 2 <= tgt.production <= 4 and neighbors > 1:
                beachhead_bonus = 1.4 
                
            distance_penalty = 1.0 / (eta ** 0.7)
            
            # Calculate ROI
            val = (tgt.production * max(0, (TOTAL_STEPS - step - eta))) * cluster_bonus * beachhead_bonus
            if tgt.id in comet_ids: val *= 0.5 # Volatile asset
            
            roi = (val / cost) * distance_penalty
            roi_matrix.append((roi, hub.id, tgt.id, int(cost), angle))

    # 5. Execute High-ROI Strikes
    roi_matrix.sort(key=lambda x: x[0], reverse=True)
    
    for roi, src_id, tgt_id, cost, angle in roi_matrix:
        if budget[src_id] >= cost:
            moves.append([src_id, angle, cost])
            budget[src_id] -= cost
            # Prevent double spending on the same target this turn
            active_flights.append({'target': tgt_id, 'eta': 1, 'owner': player, 'ships': cost})

    return moves

Writing main.py
